# D-Wave Ocean SDK Framework with Quantum Annealing

This tutorial outlines how to use the D-Wave Ocean SDK to formulate and solve a QUBO optimization problem using both a D-Wave quantum processing unit (QPU) and D-Wave's hybrid solvers. Access to the D-Wave Leap quantum cloud service is required to run problems on these remote solvers. 

**If you do not have access to the Leap quantum cloud service then check out our tutorial on [solving QUBO problems using simulated annealing]().**

## Installation & Setup



In this section, we outline how to install the D-Wave Ocean SDK. **All of these steps should be performed in a terminal opened in the directory where you downloaded or saved the notebook.** It is recommended that the Ocean SDK be installed inside a Python virtual environment rather than directly into the system. Virtual environments keep the packages used for this project isolated from other Python projects and avoid potential dependency conflicts. **[Python](https://www.python.org/downloads/) must already be installed on the machine running this demonstration.**

### Installing a Virtual Environment

The virtual environment can also be registered as a Jupyter kernel, allowing the notebook to run directly inside the same environment in which the Ocean SDK is installed.

#### macOS or Linux

To create and activate the virtual environment, run:

```bash
python3 -m venv ocean
source ocean/bin/activate
```

#### Windows Command Prompt

To create and activate the virtual environment, run:

```bat 
python -m venv ocean
ocean\Scripts\activate.bat
```

#### Windows PowerShell

To create and activate the virtual environment, run:

```powershell 
python -m venv ocean
.\ocean\Scripts\Activate.ps1
```

If PowerShell reports that *"running scripts is disabled on this system,"* run:

```powershell 
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser
```

Then retry the activation command.

### Installing Packages

Once the environment is activated, any Python packages installed with `pip` will be installed inside the ocean environment rather than globally. To install the Ocean SDK and other libraries that will be used run the commands (for any operating system),

``` bash
python -m pip install --upgrade pip

python -m pip install dwave-ocean-sdk
python -m pip install jupyter notebook ipykernel
```

Since this tutorial uses Jupyter notebooks we must register the environment with Jupyter,

``` bash
python -m ipykernel install --user --name ocean --display-name "Ocean Environment"
```

**Now you can open the Jupyter Notebook**. To open Jupyter run, 

``` bash
jupyter notebook
```

After migrating to this notebook within Jupyter **make sure to select the "Ocean Environment" kernel**.

### Authenticating D-Wave

**This section requires access to the D-Wave Leap quantum cloud service**.

Once the virtual environment is activated, the user can connect Ocean to their D-Wave Leap account by running

```bash
dwave setup --auth
```

This command authorizes Ocean to access the user's Leap account, retrieves the required Solver API (SAPI) token, and creates the D-Wave configuration file used to access remote solvers. Depending on the system, the authentication process may open a browser window and prompt the user to sign in to their Leap account.

To confirm that the configuration was successful, the user can ping an available QPU:

```bash
dwave ping --client qpu
```

This submits a small test problem to a D-Wave quantum processor and confirms that Ocean can communicate with the Leap service.

The solvers available to the user's account can also be viewed with

```bash
dwave solvers --list --all
```

This will list the QPU and hybrid solvers currently accessible through the user's Leap account.

## Review

Recall that quantum annealers are designed to solve certain classes of optimization problems, one of the most common being [Quadratic Unconstrained Binary Optimization (QUBO)](https://en.wikipedia.org/wiki/Quadratic_unconstrained_binary_optimization) problems. D-Wave's Quantum Processing Unit (QPU) represents these problems using an Ising Hamiltonian of the form

$$H(\mathbf{s}) = \sum_i h_i s_i + \sum_{i<j} J_{ij}s_i s_j,\qquad s_i\in\{-1,+1\},$$

where the coefficients $h_i$ represent individual biases and $J_{ij}$ represent interactions between pairs of spins. Using a simple change of variables between Ising spins and binary variables, typically

$$s_i = 2x_i-1,$$

we can rewrite the same problem as a QUBO,

$$Q(\mathbf{x}) = \sum_i a_i x_i + \sum_{i<j} b_{ij}x_i x_j, \qquad x_i\in\{0,1\}.$$

This allows an optimization problem written as a QUBO to be mapped directly onto the Ising model implemented by the D-Wave hardware. The QPU uses a network of superconducting qubits whose interactions are programmed according to the coefficients of the problem. During quantum annealing, the system is driven toward low-energy configurations of the Ising Hamiltonian. After measurement, these spin configurations can be converted back into binary variables, with the lowest-energy solutions corresponding to minima of the original QUBO objective function. Learn more about the physics of quantum annealing in [this]() notebook!

## Optimization Problem Statement: Max-Cut

<div style="display: flex; align-items: center; gap: 30px;">
<div style="flex: 2;">
<p>
A canonical example of a QUBO is the NP-hard <strong>Max-Cut problem</strong>.
Given a graph consisting of vertices connected by edges, the goal is to divide
the vertices into two groups such that the total weight of the edges crossing
between the groups is as large as possible. In other words, we want to find a
partition that maximizes the total weight of the edges crossing the cut.
Max-Cut is a useful example because it has a simple mathematical formulation
while still showing how a combinatorial optimization problem can be mapped onto
a QUBO and solved using the D-Wave framework.
</p>
</div>

<div style="flex: 1; text-align: center;">
<img src="https://raw.githubusercontent.com/kaufmann-group/qstep-demo/main/images/max-cut.png" style="width: 100%; max-width: 260px; height: auto;">
</div>
</div>

We assign each vertex a binary variable,

$$x_i = \begin{cases} 0 & \text{if vertex } i \text{ is in group 0},\\ 1 & \text{if vertex } i \text{ is in group 1}. \end{cases}$$

For an edge connecting vertices $i$ and $j$, the expression

$$x_i+x_j-2x_ix_j$$

is equal to $1$ when the two vertices are in different groups and $0$ when they are in the same group. This can be seen directly from the truth table, which is identical to the XOR truth table:

$$\begin{array}{c|c|c} x_i & x_j & x_i+x_j-2x_ix_j\\ \hline 0 & 0 & 0\\ 0 & 1 & 1\\ 1 & 0 & 1\\ 1 & 1 & 0 \end{array}$$

Therefore, this expression indicates whether a given edge crosses the cut. If each edge has a weight $w_{ij}$, then the total weight of all edges crossing between the two groups is

$$\sum_{(i,j)} w_{ij}\left(x_i+x_j-2x_ix_j\right).$$

The goal of the Max-Cut problem is therefore to maximize this quantity,

$$\max_{\mathbf{x}} \sum_{(i,j)} w_{ij} \left(x_i+x_j-2x_ix_j\right)$$

Because the objective is quadratic in binary variables, it can be written directly as a QUBO. Since QUBOs are conventionally expressed as minimization problems, maximizing the cut weight is equivalent to minimizing its negative:

$$H(\mathbf{x}) = -\sum_{(i,j)} w_{ij} \left(x_i+x_j-2x_ix_j\right)$$

with

$$x_i\in\{0,1\}.$$

The minimum of $H(\mathbf{x})$ therefore corresponds to the partition of the graph with the maximum total edge weight crossing between the two groups.

For this simple demonstration let's attempt to solve the max-cut problem for this weighted graph,

$$(A,B)=4,\qquad (A,C)=2,\qquad (B,C)=1,\qquad (B,D)=5,\qquad (C,D)=3.$$

In [1]:
edges = [("A", "B", 4), ("A", "C", 2), ("B", "C", 1), ("B", "D", 5), ("C", "D", 3)]

## Using D-Wave's Quantum Annealer

The input to the D-Wave system is a QUBO objective matrix in the form of a dictionary where each element represents the coefficient multiplying the $x_ix_j$ term, 

$$Q[(i,j)] = \text{coefficient multiplying } x_i x_j.$$

Using `defaultdict` we break down the expression,

$$-w_{ij}\left(x_i+x_j-2x_ix_j\right)$$

as follows.

In [2]:
from collections import defaultdict

Q = defaultdict(float)

for i, j, weight in edges:

    # -w_ij * x_i terms
    Q[(i, i)] -= weight

    # -w_ij * x_j terms
    Q[(j, j)] -= weight

    # +2 * w_ij * x_i * x_j terms
    Q[(i, j)] += 2 * weight

Q = dict(Q)

Recall that the qubits on a D-Wave QPU are not all directly connected to one another and as a result, the interaction graph of a QUBO may not map directly onto the actual connectivity of the QPU. In these cases, multiple physical qubits can be linked together in a chain to represent a single logical variable.

The process of mapping the logical QUBO graph onto the physical topology of the QPU is called minor embedding. Finding a minor embedding is computationally difficult in general, so Ocean provides tools that handle this step automatically. D-Wave's `EmbeddingComposite` uses a heuristic embedding algorithm to find a suitable mapping between the logical variables and the available physical qubits.

In [3]:
from dwave.system import DWaveSampler, EmbeddingComposite

# Connect to an available D-Wave QPU
qpu = DWaveSampler()
sampler = EmbeddingComposite(qpu)

Here, `DWaveSampler()` provides access to the physical D-Wave QPU, while `EmbeddingComposite` sits on top of the QPU sampler and automatically maps the logical QUBO onto the QPU's hardware graph before the problem is submitted.

The QUBO is submitted to the D-Wave QPU and sampled 1000 times with the `sample_qubo` function. 

In [4]:
sampleset = sampler.sample_qubo(Q, num_reads=1000)

best = sampleset.first # returns lowest energy sample in the SampleSet
solution = dict(best.sample)  

print("Solution:", solution)
print("Energy:", best.energy)

Solution: {'D': np.int8(1), 'A': np.int8(1), 'B': np.int8(0), 'C': np.int8(0)}
Energy: -14.0


Recall that `solution` is still in its binary representation, where each vertex has been assigned either `0` or `1`; so it follows to separate the vertices into the two groups.

In [5]:
group_0 = [vertex for vertex, value in solution.items() if value == 0]
group_1 = [vertex for vertex, value in solution.items() if value == 1]

print("Group 0:", group_0)
print("Group 1:", group_1)

# Solve for the max cut weight 
cut_weight = 0

for i, j, weight in edges:
    if solution[i] != solution[j]:
        cut_weight += weight

print("Cut weight:", cut_weight)

Group 0: ['B', 'C']
Group 1: ['D', 'A']
Cut weight: 14


## Using D-Wave's Hybrid Sampler

The same QUBO can also be solved directly using D-Wave's hybrid solver instead of the QPU. Unlike direct QPU sampling, the hybrid solver combines classical and quantum resources internally and outputs a single sample (as compared to a distribution like the QPU), so we do not need to manually define a minor embedding or define a sample size. 

In [6]:
from dwave.system import LeapHybridSampler

sampler_hybrid = LeapHybridSampler()
sampleset_hybrid = sampler_hybrid.sample_qubo(Q, time_limit=10)

best_hybrid = sampleset_hybrid.first
solution_hybrid = dict(best_hybrid.sample)

print("Solution:", solution_hybrid)
print("Energy:", best_hybrid.energy)

Solution: {'A': np.int8(1), 'B': np.int8(0), 'C': np.int8(0), 'D': np.int8(1)}
Energy: -14.0


## Problem Inspector:

D-Wave has created a way for users to look at both the logical QUBO graph submitted to the QPU and the QPU itself alongside the minor embedding. This is best demonstrated for the large graph max-cut problem given below.

In [7]:
many_edges = [("A", "B", 4), ("A", "C", 7), ("A", "E", 3), ("A", "F", 8), ("A", "H", 5), ("B", "C", 2), ("B", "D", 6), ("B", "G", 9), ("B", "I", 4), ("C", "D", 5), ("C", "E", 8), ("C", "H", 6), ("C", "J", 3), ("D", "F", 7), ("D", "G", 4), ("D", "J", 9), ("D", "K", 2), ("E", "F", 5), ("E", "H", 7), ("E", "L", 6), ("F", "G", 3), ("F", "I", 8), ("F", "M", 4), ("G", "H", 6), ("G", "J", 5), ("G", "N", 9), ("H", "I", 2), ("H", "K", 8), ("H", "O", 4), ("I", "J", 7), ("I", "L", 5), ("I", "P", 6), ("J", "K", 3), ("J", "M", 8), ("J", "Q", 5), ("K", "L", 7), ("K", "N", 4), ("K", "R", 9), ("L", "M", 2), ("L", "O", 6), ("L", "S", 8), ("M", "N", 5), ("M", "P", 7), ("M", "T", 3), ("N", "O", 8), ("N", "Q", 6), ("O", "P", 4), ("O", "R", 7), ("P", "Q", 9), ("P", "S", 5), ("Q", "R", 3), ("Q", "T", 8), ("R", "S", 6), ("S", "T", 7)]

vertices = sorted(set(vertex for i, j, weight in many_edges for vertex in (i, j)))

print("Number of vertices:", len(vertices))
print("Number of edges:", len(edges))

Number of vertices: 20
Number of edges: 5


In [8]:
import dimod
import dwave.inspector 

Q_many = defaultdict(float)

for i, j, weight in many_edges:

    # -w_ij * x_i terms
    Q_many[(i, i)] -= weight

    # -w_ij * x_j terms
    Q_many[(j, j)] -= weight

    # +2 * w_ij * x_i * x_j terms
    Q_many[(i, j)] += 2 * weight

bqm = dimod.BinaryQuadraticModel.from_qubo(dict(Q_many))

In [ ]:
sampler = EmbeddingComposite(DWaveSampler())
sampleset = sampler.sample(bqm, num_reads=1000)

dwave.inspector.show(sampleset)

Finished? Check out our other tutorials! 

<a class="notebook-download"
   href="./downloads/quantum_annealing_notebook.txt"
   download="D-wave SDK Demonstration Quantum Annealing.ipynb">
  Download notebook ↓
</a>